In [8]:
!pip install xgboost

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib 
from pathlib import Path
import warnings
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import pickle

In [10]:
# Creates a path that works on any operating system
data_path = Path('backend') / 'data' / 'HRDataset_v14.csv'
df = pd.read_csv(data_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311 entries, 0 to 310
Data columns (total 36 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Employee_Name               311 non-null    object 
 1   EmpID                       311 non-null    int64  
 2   MarriedID                   311 non-null    int64  
 3   MaritalStatusID             311 non-null    int64  
 4   GenderID                    311 non-null    int64  
 5   EmpStatusID                 311 non-null    int64  
 6   DeptID                      311 non-null    int64  
 7   PerfScoreID                 311 non-null    int64  
 8   FromDiversityJobFairID      311 non-null    int64  
 9   Salary                      311 non-null    int64  
 10  Termd                       311 non-null    int64  
 11  PositionID                  311 non-null    int64  
 12  Position                    311 non-null    object 
 13  State                       311 non

In [11]:
current_date = datetime.now()

In [12]:
df['DOB'] = pd.to_datetime(df['DOB'], format='%m/%d/%y', errors='coerce')

In [13]:
#Ignore UserWarnings because the DOB date-string format is inconsistent
#Some data might be have '1987' as the year and other might have '87', which causes warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [14]:
df['DOB'] = pd.to_datetime(df['DOB'], errors='coerce')
df['Age'] = df['DOB'].apply(lambda x: (current_date - x).days // 365)

In [15]:
df['Age']

0      42
1      50
2      37
3      37
4      36
       ..
306    40
307    43
308    46
309    47
310    47
Name: Age, Length: 311, dtype: int64

<h4>In the above cell, we can see that an age is in the negatives</h4>

- It occurs when year in DOB is given as '87' instead of '1987'. The '87' is considered as '2087'.
- A function (fix_2digit_bug) is created to deal with that error.

<h5>Handling anomalies</h5>

In [16]:
def fix_2digit_bug(date):
    if date.year > current_date.year:
        return date.replace(year=date.year - 100)
    return date

#Logic
#2087-100=1987

<h4>Feature engineering: Employee Age</h4>

In [17]:
df['DOB'] = df['DOB'].apply(fix_2digit_bug)
df['Age'] = (current_date - df['DOB']).dt.days // 365

df['Age']

0      42
1      50
2      37
3      37
4      36
       ..
306    40
307    43
308    46
309    47
310    47
Name: Age, Length: 311, dtype: int64

<h4>Feature engineering: Tenure Calculation</h4>

In [18]:
df['DateofHire'] = pd.to_datetime(df['DateofHire'])
df['DateofTermination'] = pd.to_datetime(df['DateofTermination'])


#Vectorized code. It is an IF condition.
#IF DateofTermination is missing, then employee still works in the organization. So we calculate tenure till current day.
df['Tenure'] = np.where(
    df['DateofTermination'].isna(),
    (current_date - df['DateofHire']).dt.days / 365,
    (df['DateofTermination'] - df['DateofHire']).dt.days / 365
)


print(df[['Employee_Name', 'Tenure', 'DateofTermination']])

                Employee_Name     Tenure DateofTermination
0         Adinolfi, Wilson  K  14.673973               NaT
1    Ait Sidi, Karthikeyan      1.216438        2016-06-16
2           Akinkuolie, Sarah   1.224658        2012-09-24
3                Alagbe,Trina  18.167123               NaT
4            Anderson, Carol    5.161644        2016-09-06
..                        ...        ...               ...
306            Woodson, Jason  11.665753               NaT
307        Ybarra, Catherine    7.076712        2015-09-29
308          Zamora, Jennifer  15.909589               NaT
309               Zhou, Julia  10.936986               NaT
310             Zima, Colleen  11.435616               NaT

[311 rows x 3 columns]


In [19]:
df['RaceDesc']

0      White
1      White
2      White
3      White
4      White
       ...  
306    White
307    Asian
308    White
309    White
310    Asian
Name: RaceDesc, Length: 311, dtype: object

In [20]:
# Show columns that might hide more features
print(df[['Position', 'State', 'MaritalDesc', 'ManagerName', 'RecruitmentSource', 'SpecialProjectsCount', 'Absences']].head(10))

# Also check for unique counts to see if a column is too messy to encode
print("\nUnique values in Position:", df['Position'].nunique())

                   Position State MaritalDesc      ManagerName  \
0   Production Technician I    MA      Single   Michael Albert   
1                   Sr. DBA    MA     Married       Simon Roup   
2  Production Technician II    MA     Married   Kissy Sullivan   
3   Production Technician I    MA     Married     Elijiah Gray   
4   Production Technician I    MA    Divorced   Webster Butler   
5   Production Technician I    MA      Single         Amy Dunn   
6         Software Engineer    MA      Single  Alex Sweetwater   
7   Production Technician I    MA     Widowed    Ketsia Liebig   
8   Production Technician I    MA      Single   Brannon Miller   
9                IT Support    MA    Divorced     Peter Monroe   

    RecruitmentSource  SpecialProjectsCount  Absences  
0            LinkedIn                     0         1  
1              Indeed                     6        17  
2            LinkedIn                     0         3  
3              Indeed                     0      


<h5>Explicitly removing protected class variables (Race/Ethnicity) and redundant columns to prevent algorithmic bias</h5>

In [21]:
cols_to_drop = ['HispanicLatino', 'RaceDesc', 'CitizenDesc', 'Employee_Name', 'EmpID', 'MaritalDesc']
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 311 entries, 0 to 310
Data columns (total 32 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   MarriedID                   311 non-null    int64         
 1   MaritalStatusID             311 non-null    int64         
 2   GenderID                    311 non-null    int64         
 3   EmpStatusID                 311 non-null    int64         
 4   DeptID                      311 non-null    int64         
 5   PerfScoreID                 311 non-null    int64         
 6   FromDiversityJobFairID      311 non-null    int64         
 7   Salary                      311 non-null    int64         
 8   Termd                       311 non-null    int64         
 9   PositionID                  311 non-null    int64         
 10  Position                    311 non-null    object        
 11  State                       311 non-null    object        

<h5>Label Encoding</h5>

In [23]:
#.strip() to remove trailing whitespace
#.map() for label encoding

if df['Sex'].dtype == 'object': ## Only run the encoding if 'Sex' is still text/object (prevents errors if cell is rerun)
    df['Sex'] = df['Sex'].str.strip().map({'M': 0, 'F': 1})

df['Sex']

0      0
1      0
2      1
3      1
4      1
      ..
306    0
307    1
308    1
309    1
310    1
Name: Sex, Length: 311, dtype: int64

<h4>One Hot Encoding</h4>

- Convert text categories into binary (0/1) columns to make the data machine-readable.

In [24]:
cols_to_encode = ['Department', 'Position', 'RecruitmentSource', 'PerformanceScore', 'EmploymentStatus']

df_final = pd.get_dummies(df, columns=cols_to_encode, drop_first=True)
# drop_first=True removes one category to prevent redundant data (multicollinearity). 

In [25]:
df_final

,MarriedID,MaritalStatusID,GenderID,EmpStatusID,DeptID,PerfScoreID,FromDiversityJobFairID,Salary,Termd,PositionID,...,RecruitmentSource_Indeed,RecruitmentSource_LinkedIn,RecruitmentSource_On-line Web application,RecruitmentSource_Other,RecruitmentSource_Website,PerformanceScore_Fully Meets,PerformanceScore_Needs Improvement,PerformanceScore_PIP,EmploymentStatus_Terminated for Cause,EmploymentStatus_Voluntarily Terminated
0,0,0,1,1,5,4,0,62506,0,19,...,False,True,False,False,False,False,False,False,False,False
1,1,1,1,5,3,3,0,104437,1,27,...,True,False,False,False,False,True,False,False,False,True
2,1,1,0,5,5,3,0,64955,1,20,...,False,True,False,False,False,True,False,False,False,True
3,1,1,0,1,5,3,0,64991,0,19,...,True,False,False,False,False,True,False,False,False,False
4,0,2,0,5,5,3,0,50825,1,19,...,False,False,False,False,False,True,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306,0,0,1,1,5,3,0,65893,0,20,...,False,True,False,False,False,True,False,False,False,False
307,0,0,0,5,5,1,0,48513,1,19,...,False,False,False,False,False,False,False,True,False,True
308,0,0,0,1,3,4,0,220450,0,6,...,False,False,False,False,False,False,False,False,False,False
309,0,0,0,1,3,3,0,89292,0,9,...,False,False,False,False,False,True,False,False,False,False


In [26]:
print(f"\nPrevious total columns: {len(df.columns)}")
print(f"\nNew total columns: {len(df_final.columns)}")


Previous total columns: 32

New total columns: 76


In [27]:
df_final.columns

Index(['MarriedID', 'MaritalStatusID', 'GenderID', 'EmpStatusID', 'DeptID',
       'PerfScoreID', 'FromDiversityJobFairID', 'Salary', 'Termd',
       'PositionID', 'State', 'Zip', 'DOB', 'Sex', 'DateofHire',
       'DateofTermination', 'TermReason', 'ManagerName', 'ManagerID',
       'EngagementSurvey', 'EmpSatisfaction', 'SpecialProjectsCount',
       'LastPerformanceReview_Date', 'DaysLateLast30', 'Absences', 'Age',
       'Tenure', 'Department_Executive Office', 'Department_IT/IS',
       'Department_Production       ', 'Department_Sales',
       'Department_Software Engineering', 'Position_Administrative Assistant',
       'Position_Area Sales Manager', 'Position_BI Developer',
       'Position_BI Director', 'Position_CIO', 'Position_Data Analyst',
       'Position_Data Analyst ', 'Position_Data Architect',
       'Position_Database Administrator', 'Position_Director of Operations',
       'Position_Director of Sales', 'Position_Enterprise Architect',
       'Position_IT Director',

<h5>Now, columns that have become noise or useless for the model are dropped.</h5>

In [28]:
noise = [
    'MarriedID', 'MaritalStatusID', 'GenderID', 'EmpStatusID', 'DeptID',
    'PerfScoreID', 'FromDiversityJobFairID', 'PositionID', 'State', 'Zip', 
    'DOB', 'DateofHire', 'DateofTermination', 'TermReason', 'ManagerID', 
    'ManagerName', 'LastPerformanceReview_Date', 
    'EmploymentStatus_Terminated for Cause', 'EmploymentStatus_Voluntarily Terminated'
]
#Remove 'TermReason' and 'EmploymentStatus' to prevent Target Leakage.

df_ml = df_final.drop(columns=noise, errors='ignore')

In [29]:
df_ml.columns

Index(['Salary', 'Termd', 'Sex', 'EngagementSurvey', 'EmpSatisfaction',
       'SpecialProjectsCount', 'DaysLateLast30', 'Absences', 'Age', 'Tenure',
       'Department_Executive Office', 'Department_IT/IS',
       'Department_Production       ', 'Department_Sales',
       'Department_Software Engineering', 'Position_Administrative Assistant',
       'Position_Area Sales Manager', 'Position_BI Developer',
       'Position_BI Director', 'Position_CIO', 'Position_Data Analyst',
       'Position_Data Analyst ', 'Position_Data Architect',
       'Position_Database Administrator', 'Position_Director of Operations',
       'Position_Director of Sales', 'Position_Enterprise Architect',
       'Position_IT Director', 'Position_IT Manager - DB',
       'Position_IT Manager - Infra', 'Position_IT Manager - Support',
       'Position_IT Support', 'Position_Network Engineer',
       'Position_President & CEO', 'Position_Principal Data Architect',
       'Position_Production Manager', 'Position_Pro

- Column 'Position_Data Analyst' seems to have a duplicate, the duplicate has a trailing space in its column name
- So, the two columns are summed together into one column (0+1), and the duplicate with the trailing space is dropped.

In [30]:
if 'Position_Data Analyst ' in df_ml.columns:
    df_ml['Position_Data Analyst'] = df_ml['Position_Data Analyst'] + df_ml['Position_Data Analyst ']
    df_ml.drop(columns=['Position_Data Analyst '], inplace=True)


df_ml.columns

Index(['Salary', 'Termd', 'Sex', 'EngagementSurvey', 'EmpSatisfaction',
       'SpecialProjectsCount', 'DaysLateLast30', 'Absences', 'Age', 'Tenure',
       'Department_Executive Office', 'Department_IT/IS',
       'Department_Production       ', 'Department_Sales',
       'Department_Software Engineering', 'Position_Administrative Assistant',
       'Position_Area Sales Manager', 'Position_BI Developer',
       'Position_BI Director', 'Position_CIO', 'Position_Data Analyst',
       'Position_Data Architect', 'Position_Database Administrator',
       'Position_Director of Operations', 'Position_Director of Sales',
       'Position_Enterprise Architect', 'Position_IT Director',
       'Position_IT Manager - DB', 'Position_IT Manager - Infra',
       'Position_IT Manager - Support', 'Position_IT Support',
       'Position_Network Engineer', 'Position_President & CEO',
       'Position_Principal Data Architect', 'Position_Production Manager',
       'Position_Production Technician I', 'Po

<h5>x -> Features</h5>
<h5>y -> Target</h5>

In [31]:
y = df_ml['Termd']

X = df_ml.drop(columns=['Termd','Tenure']) #Set feature set to everything but the target

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#test_size=0.2, 20% of the dataset is kept aside for testing and the rest (80%) are used for training.
#random_state=42, ensures that random shuffle is same every time the code is run.
#The function returns four separate blocks of data. 

print(f"Training set size: {X_train.shape[0]} employees")
print(f"Testing set size: {X_test.shape[0]} employees")

Training set size: 248 employees
Testing set size: 63 employees


<h4>SMOTE (Synthetic Minority Oversampling Technique)</h4>

-  It aims to balance class distribution by randomly increasing minority class examples by replicating them.
-  Since more people stay than leave. If 90% stay, an AI can cheat by just guessing "Stay" every time and be 90% accurate.
-  The number of people who have been terminated (Termd = 1) is much smaller than those who are still employed (Termd = 0).
-  If model is trained on this "unbalanced" data, it will have High Accuracy but Low Recall. (more false positives).
-  Synthetic data is created on the training set, so SMOTE must be done after splitting.
  

- Initial experiments with SMOTE yielded an accuracy of approximately 66-67%. While SMOTE balanced the classes, it introduced synthetic "noise" that limited the model's precision.
- SMOTE creates synthetic examples by interpolating between existing minority points. In a small dataset like this one, this lead to Over-Generalization, leading to a drop in real-world precision.
- So, I did pivot to Cost Sensitive Learning

In [33]:
sm = SMOTE(random_state=42)

# Create a balanced version of training data
# Generates synthetic/fake data of employees who left to match the number of those who stayed
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(f"Original 'Left' count: {y_train.sum()}")
print(f"Balanced 'Left' count: {y_train_res.sum()}")
#When sum() is used on a column of 0s and 1s, it only counts the 1s. So, 82 is the number of people who Left in OG training set and 166 is the new synthetic data.

Original 'Left' count: 82
Balanced 'Left' count: 166


## Competitive Model Benchmarking

<h3>Logistic Regression</h3>

<h4>With SMOTE</h4>

In [82]:
log_model = LogisticRegression(max_iter=1000, random_state=42)
#random_state freezes the randomness. Ensures reproducibility of results.

#Train model on the BALANCED training data
log_model.fit(X_train_res, y_train_res)

#Predict on the REAL (unbalanced) test data
log_preds = log_model.predict(X_test)

print("="*50)
print("LOGISTIC REGRESSION w/ SMOTE")
print(classification_report(y_test, log_preds))
#y_test: The real answer of who stayed and left
#log_preds: What the Logistic Regression guessed

LOGISTIC REGRESSION w/ SMOTE
              precision    recall  f1-score   support

           0       0.72      0.71      0.72        41
           1       0.48      0.50      0.49        22

    accuracy                           0.63        63
   macro avg       0.60      0.60      0.60        63
weighted avg       0.64      0.63      0.64        63



<h4>Without SMOTE</h4>

log_model1 = LogisticRegression(class_weight={0: 1, 1: 1.5},max_iter=1000, random_state=42)

log_model1.fit(X_train, y_train)

log_preds1 = log_model1.predict(X_test)

print("="*50)
print("LOGISTIC REGRESSION w/o SMOTE")
print(classification_report(y_test, log_preds1))

<h3>Random Forest</h3>

<h4>With SMOTE</h4>

In [35]:
#Random Forest creates multiple decision trees that act as a council and 'vote' on a prediction. Here, 100 such 'trees' are made.

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train_res, y_train_res)

rf_preds = rf_model.predict(X_test)

print("="*50)
print("RANDOM FOREST")
print(classification_report(y_test, rf_preds))

RANDOM FOREST
              precision    recall  f1-score   support

           0       0.72      0.83      0.77        41
           1       0.56      0.41      0.47        22

    accuracy                           0.68        63
   macro avg       0.64      0.62      0.62        63
weighted avg       0.67      0.68      0.67        63



<h4>Without SMOTE</h4>

In [84]:
rf_model1 = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced_subsample', max_depth=5)

rf_model1.fit(X_train, y_train)

rf_preds1 = rf_model1.predict(X_test)

print("="*50)
print("RANDOM FOREST")
print(classification_report(y_test, rf_preds1))

RANDOM FOREST
              precision    recall  f1-score   support

           0       0.75      0.80      0.78        41
           1       0.58      0.50      0.54        22

    accuracy                           0.70        63
   macro avg       0.66      0.65      0.66        63
weighted avg       0.69      0.70      0.69        63



<h3>XGBoost</h3>

<h4>With SMOTE</h4>

In [70]:
#XGBoost builds the decision trees one after another. So, Tree #2 learns from mistake made by Tree #1 and so on.
old_xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1.0, 
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
old_xgb_model.fit(X_train_res, y_train_res)

old_xgb_preds = old_xgb_model.predict(X_test)

print("="*50)
print("OLD XGBOOST w/ SMOTE")
print(classification_report(y_test, old_xgb_preds))

#ACCURACY
accuracy = accuracy_score(y_test, old_xgb_preds)

print(f"Model Accuracy: {accuracy * 100:.2f}%")

OLD XGBOOST w/ SMOTE
              precision    recall  f1-score   support

           0       0.72      0.80      0.76        41
           1       0.53      0.41      0.46        22

    accuracy                           0.67        63
   macro avg       0.62      0.61      0.61        63
weighted avg       0.65      0.67      0.65        63

Model Accuracy: 66.67%


<h4>Without SMOTE</h4>

In [100]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=1.5, 
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_test)

print("="*50)
print("XGBOOST")
print(classification_report(y_test, xgb_preds))

#ACCURACY
accuracy = accuracy_score(y_test, xgb_preds)

print(f"Model Accuracy: {accuracy * 100:.2f}%")

XGBOOST
              precision    recall  f1-score   support

           0       0.73      0.85      0.79        41
           1       0.60      0.41      0.49        22

    accuracy                           0.70        63
   macro avg       0.66      0.63      0.64        63
weighted avg       0.68      0.70      0.68        63

Model Accuracy: 69.84%


In [85]:
importances = xgb_model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

print(feature_importance_df.head(10))

                                 Feature  Importance
45       RecruitmentSource_Google Search    0.081475
38            Position_Software Engineer    0.058558
43  RecruitmentSource_Diversity Job Fair    0.049941
10          Department_Production           0.049385
4                   SpecialProjectsCount    0.048424
47            RecruitmentSource_LinkedIn    0.042264
5                         DaysLateLast30    0.042180
46              RecruitmentSource_Indeed    0.041881
34     Position_Production Technician II    0.037722
14           Position_Area Sales Manager    0.037675


<h3>XGBoost has been identified as the optimal production model, achieving a peak accuracy of 69.84% (70%).</h3>

In [75]:
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

results_df = pd.DataFrame({
    'Actual_Status': y_test,
    'Predicted_Status': xgb_preds,
    'Attrition_Risk_%': (xgb_probs * 100).round(2) # Convert 0.85 to 85.0%
})

print(results_df.sort_values(by='Attrition_Risk_%', ascending=False).head(10))

     Actual_Status  Predicted_Status  Attrition_Risk_%
92               0                 1         69.449997
84               1                 1         65.919998
242              1                 1         63.720001
224              1                 1         62.450001
303              1                 1         60.650002
93               1                 1         60.639999
24               1                 1         60.099998
251              0                 1         59.910000
17               0                 1         59.779999
226              1                 1         55.509998


## Data Leakage Audit 

During initial model testing for the first model (Logistic Regression), the model achieved **100% Accuracy (F1-score: 1.00)**. 
Upon investigation using Random Forest's **Feature Importance** analysis, it was discovered that the `Tenure` 
variable was acting as a 'Leaky Feature.'. This 'Tenure' information is not available for current employees (the target), 
hence including it allowed the model to 'cheat' by looking at the timeline of the exit rather 
than the behavioral drivers of the exit.

In [76]:
joblib.dump(xgb_model, 'attrition_model.pkl')

#Save Column Order is essential for the API as model expects columns to be in same order.
model_columns = list(X.columns)
joblib.dump(model_columns, 'model_columns.pkl')

['model_columns.pkl']